In [2]:
import pandas as pd
import numpy as np
from collections import namedtuple
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
import onekey_algo.custom.components as okcomp
from onekey_algo.custom.components.comp1 import fillna
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300

In [ ]:

data_file = r''
labels = ['label']
featrues_not_use = ['ID']

structed_data = pd.read_csv(data_file, header=0)
structed_data

In [ ]:

ids = structed_data['ID']
structed_data = structed_data.drop(featrues_not_use, axis=1)
structed_data

In [ ]:
structed_data.describe()

In [ ]:

data = structed_data
data

In [ ]:

spearman_corr = data[[c for c in data.columns if c !='IM']].corr('spearman')

import seaborn as sns
import matplotlib.pyplot as plt
from onekey_algo.custom.components.comp1 import draw_matrix


In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp

n_classes = 2
train_data = data[(data['group'] == 'train')]
train_ids = ids.loc[list(train_data.index)]
train_data = train_data.reset_index()
train_data = train_data.drop('index', axis=1)
y_data = train_data[labels]
X_data = train_data.drop(labels + ['group'], axis=1)

val_data = data[data['group'] == 'val']
val_ids = ids.loc[list(val_data.index)]
val_data = val_data.reset_index()
val_data = val_data.drop('index', axis=1)
y_val_data = val_data[labels]
X_val_data = val_data.drop(labels + ['group'], axis=1)

test_data = data[data['group'] == 'test']
test_ids = ids.loc[list(test_data.index)]
test_data = test_data.reset_index()
test_data = test_data.drop('index', axis=1)
y_test_data = test_data[labels]
X_test_data = test_data.drop(labels + ['group'], axis=1)

y_all_data = data[labels]
X_all_data = data.drop(labels + ['group'], axis=1)

column_names = X_data.columns
print(f"{X_data.shape}, {X_val_data.shape}, {X_test_data.shape}")


In [ ]:
y_test_data.value_counts()

In [ ]:
model_names = ['LR']
models = okcomp.comp1.create_clf_model_none_overfit(model_names)
model_names = list(models.keys())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score


results = okcomp.comp1.get_bst_split(X_data, y_data, models, test_size=0.2, metric_fn=roc_auc_score, n_trails=5, cv=True, random_state=0)
_, (X_train_sel, X_test_sel, y_train_sel, y_test_sel) = results['results'][results['max_idx']]
X_train_sel, X_val_sel, X_test_sel, y_train_sel, y_val_sel, y_test_sel = X_data, X_val_data, X_test_data, y_data, y_val_data, y_test_data
trails, _ = zip(*results['results'])
cv_results = pd.DataFrame(trails, columns=model_names)


In [ ]:
targets = []
for l in labels:
    new_models = list(okcomp.comp1.create_clf_model_none_overfit(model_names).values())
    for m in new_models:
        m.fit(X_train_sel, y_train_sel[l])
        y_pred = m.predict(X_test_sel)
    targets.append(new_models)

In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportion_confint
from onekey_algo.custom.components.metrics import analysis_pred_binary


def calculate_acc_ci(acc, n, alpha=0.05):
    lower, upper = proportion_confint(acc * n, n, alpha=alpha, method='beta')
    return f"{lower:.4f} - {upper:.4f}"

predictions = [[(model.predict(X_train_sel), model.predict(X_val_sel), model.predict(X_test_sel)) 
                for model in target] for label, target in zip(labels, targets)]
pred_scores = [[(model.predict_proba(X_train_sel), model.predict_proba(X_val_sel), model.predict_proba(X_test_sel)) 
                for model in target] for label, target in zip(labels, targets)]

metric = []
pred_sel_idx = []
for label, prediction, scores in zip(labels, predictions, pred_scores):
    pred_sel_idx_label = []
    for mname, (train_pred, val_pred, test_pred), (train_score, val_score, test_score) in zip(model_names, prediction, scores):

        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_train_sel[label], 
                                                                                              train_score[:, 1])
        acc_ci = calculate_acc_ci(acc, len(y_train_sel[label]))  # 计算Acc的95%CI
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, acc_ci, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, "Train"))
                 

        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_val_sel[label], 
                                                                                             val_score[:, 1])
        acc_ci = calculate_acc_ci(acc, len(y_val_sel[label]))  # 计算Acc的95%CI
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, acc_ci, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, "Val"))
        

        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_test_sel[label], 
                                                                                              test_score[:, 1])
        acc_ci = calculate_acc_ci(acc, len(y_test_sel[label]))  # 计算Acc的95%CI
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, acc_ci, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, "Test"))
        

        pred_sel_idx_label.append(np.logical_or(test_score[:, 0] >= thres, test_score[:, 1] >= thres))
    
    pred_sel_idx.append(pred_sel_idx_label)
    
metric_df = pd.DataFrame(metric, index=None, columns=['model_name', 'Acc', 'Acc 95%CI', 'AUC', 'AUC 95%CI',
                                                      'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Precision', 
                                                      'Recall', 'F1', 'Threshold', 'Cohort'])

metric_df.to_csv('G:\\.DLR\\sol11_plus\\combined_results\\.2.5D_combined.csv', index=False)
metric_df

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from onekey_algo.custom.components.delong import calc_95_CI
from onekey_algo.custom.components.metrics import analysis_pred_binary

predictions = [[(model.predict(X_train_sel), model.predict(X_val_sel), model.predict(X_test_sel)) 
                for model in target] for label, target in zip(labels, targets)]
pred_scores = [[(model.predict_proba(X_train_sel), model.predict_proba(X_val_sel), model.predict_proba(X_test_sel)) 
                for model in target] for label, target in zip(labels, targets)]

metric = []
pred_sel_idx = []
for label, prediction, scores in zip(labels, predictions, pred_scores):
    pred_sel_idx_label = []
    for mname, (train_pred, val_pred, test_pred), (train_score, val_score, test_score) in zip(model_names, prediction, scores):
        # 计算训练集指数
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_train_sel[label], 
                                                                                              train_score[:, 1])
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"Train"))
                 
        # 计算验证集指标
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_val_sel[label], 
                                                                                             val_score[:, 1])
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"Val"))
        
        # 计算测试集指标
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_test_sel[label], 
                                                                                              test_score[:, 1])
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"Test"))
        
        # 计算thres对应的sel idx
        pred_sel_idx_label.append(np.logical_or(test_score[:, 0] >= thres, test_score[:, 1] >= thres))
    
    pred_sel_idx.append(pred_sel_idx_label)
metric = pd.DataFrame(metric, index=None, columns=['model_name', 'Accuracy', 'AUC', '95% CI',
                                                   'Sensitivity', 'Specificity', 
                                                   'PPV', 'NPV', 'Precision', 'Recall', 'F1',
                                                   'Threshold', 'Cohort'])
metric

In [ ]:
import seaborn as sns

plt.subplot(211)
sns.barplot(x='model_name', y='Accuracy', data=metric, hue='Cohort')
plt.subplot(212)
sns.lineplot(x='model_name', y='Accuracy', data=metric, hue='Cohort')


In [ ]:
import os
import numpy as np

sel_model = model_names
os.makedirs('results', exist_ok=True)
for idx, label in enumerate(labels):
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            target = targets[idx][sel_model_idx]

            train_indexes = np.reshape(np.array(train_ids), (-1, 1)).astype(str)
            test_indexes = np.reshape(np.array(test_ids), (-1, 1)).astype(str)
            val_indexes = np.reshape(np.array(val_ids), (-1, 1)).astype(str)
            y_train_pred_scores = target.predict_proba(X_train_sel)
            y_test_pred_scores = target.predict_proba(X_test_sel)
            y_val_pred_scores = target.predict_proba(X_val_sel)
            columns = ['ID'] + [f"{label}-{i}"for i in range(y_test_pred_scores.shape[1])]

            result_train = pd.DataFrame(np.concatenate([train_indexes, y_train_pred_scores], axis=1), columns=columns)
            result_train.to_csv(f'./results/2.5D_combined_{sm}_train.csv', index=False)
            result_val = pd.DataFrame(np.concatenate([val_indexes, y_val_pred_scores], axis=1), columns=columns)
            result_val.to_csv(f'./results/2.5D_combined_{sm}_val.csv', index=False)
            result_test = pd.DataFrame(np.concatenate([test_indexes, y_test_pred_scores], axis=1), columns=columns)
            result_test.to_csv(f'./results/2.5D_combined_{sm}_test.csv', index=False)